# Volume 11, Chapter 4: The Itô integral (Python companion)

Exercises first, solutions at the end. Itô against Stratonovich sums, Itô's formula checked path by path, the local time computed three ways, Lévy's theorem $L_t\overset{d}{=}|B_t|$, a martingale turned back into Brownian motion by its own clock (Dambis–Dubins–Schwarz), Burkholder's inequality, and a computer science application: **estimating the local time from sampled data**, the kernel density problem for occupation measures, with its bandwidth trade-off. Requires `numpy`.

In [ ]:
import math
import numpy as np
rng = np.random.default_rng(1104)

def bm(T, N, m=1):
    # m Brownian paths on [0, T] at N + 1 equally spaced times
    steps = rng.standard_normal((m, N)) * math.sqrt(T / N)
    return np.concatenate([np.zeros((m, 1)), np.cumsum(steps, axis=1)], axis=1)

## Exercises

**P4.1 (★) Where to evaluate the integrand.** On one path on $[0,1]$ with $2^{16}$ steps, compute the left, midpoint and right sums for $\int_0^1B\,\mathrm dB$ along the dyadic partitions of levels $4,8,12,16$, and compare with $\frac12(B_1^2-1)$, $\frac12B_1^2$, $\frac12(B_1^2+1)$ (Exercise 4.1). Over $1000$ paths, check that only the left sums have mean $0$.

**P4.2 (★) Itô's formula, path by path.** For $f(x)=\sin x$, compute $f(B_1)-\sum f'(B_{t_{i-1}})\Delta B_i-\frac12\sum f''(B_{t_{i-1}})\Delta t$ on one path with $2^n$ steps, $n=6,\dots,18$. Theorem 4.11 says it tends to $f(0)=0$. At what rate? Do the same without the correction term.

**P4.3 (★★) The local time three ways.** On a path on $[0,1]$ with $2^{18}$ steps compute (i) $|B_1|-\sum\operatorname{sgn}(B_{t_{i-1}})\Delta B_i$ (Tanaka, Theorem 4.19), (ii) $\frac1{2\varepsilon}\lambda\{s\le1:|B_s|<\varepsilon\}$ for $\varepsilon=0.05,0.02,0.01,0.005$ (Exercise 4.6), (iii) $\max_{s\le1}(-\beta_s)$ (Skorokhod, Lemma 4.20). Then over $4000$ paths compare the law of $L_1$ with that of $|B_1|$ (Theorem 4.21).

**P4.4 (★★) Undoing a clock.** Let $M_t=\int_0^tB_s\,\mathrm dB_s=\frac12(B_t^2-t)$, with clock $\langle M\rangle_t=\int_0^tB_s^2\,\mathrm ds$. Over $2000$ paths on $[0,8]$, sample $\beta_s=M_{T_s}$ at $s=0.5$ and $s=1$ (Theorem 4.16). Check that $\beta_1$ is $N(0,1)$ (mean, variance, fourth moment) and that $\beta_1-\beta_{1/2}$ is uncorrelated with $\beta_{1/2}$ and with $\beta_{1/2}^2$, although $M_t$ itself is far from Gaussian.

**P4.5 (★★) Burkholder's constant.** For $p=4$, Theorem 4.18 gives $\mathbb E\sup_{s\le t}M_s^4\le C_4\,\mathbb E\langle M\rangle_t^2$ with $C_4=6^2(4/3)^8\approx360$. Estimate the ratio $\mathbb E\sup M^4/\mathbb E\langle M\rangle^2$ for $M=B$ and for $M=\int B\,\mathrm dB$ on $[0,1]$. How far from sharp is the constant?

**P4.6 (★★★) CS application: local time from sampled data.** A sensor records a process at $n$ equally spaced times on $[0,1]$, and we want the time it spends near a level, per unit of space: the local time. The natural estimator is the kernel density estimate of the occupation measure, $\hat L=\frac1{2h}\cdot\frac1n\#\{i:|B_{i/n}|<h\}$. Too small a bandwidth $h$ sees too few samples; too large a bandwidth blurs. Using a fine path ($2^{20}$ steps) for the true $L_1$, find the best $h$ for $n=2^8,2^{10},2^{12},2^{14}$ over $60$ paths, and compare it with the typical step $\sqrt{1/n}$.

## Solutions

### P4.1
**How to find it.** The right sum minus the left sum is $\sum(\Delta B_i)^2$, which tends to $1$; the midpoint sum telescopes exactly to $\frac12B_1^2$ at every level. The left sums are discrete martingale transforms, hence have mean exactly $0$ at every level.

In [ ]:
n = 16
B = bm(1.0, 2**n)[0]
for lev in (4, 8, 12, 16):
    b = B[::2**(n - lev)]; d = np.diff(b)
    print(f"level {lev:2d}: left {np.sum(b[:-1]*d): .4f}  mid {np.sum(0.5*(b[:-1]+b[1:])*d): .4f}  right {np.sum(b[1:]*d): .4f}")
print(f"targets : left {0.5*(B[-1]**2-1): .4f}  mid {0.5*B[-1]**2: .4f}  right {0.5*(B[-1]**2+1): .4f}")
P = bm(1.0, 2**10, 1000); d = np.diff(P, axis=1)
print("means over 1000 paths: left", round(float(np.mean(np.sum(P[:, :-1]*d, axis=1))), 3),
      " mid", round(float(np.mean(np.sum(0.5*(P[:, :-1]+P[:, 1:])*d, axis=1))), 3),
      " right", round(float(np.mean(np.sum(P[:, 1:]*d, axis=1))), 3))

### P4.2
**How to find it.** Without the correction the defect tends to $-\frac12\int_0^1f''(B_s)\,\mathrm ds=\frac12\int_0^1\sin B_s\,\mathrm ds$, which is not zero. With it, the error comes from the third-order Taylor terms and from the Riemann sum of the correction; on this path it decreases to $0$, with fluctuations, and stays below the square root of the step, $2^{-n/2}$.

In [ ]:
B = bm(1.0, 2**18)[0]
for n in range(6, 19, 2):
    b = B[::2**(18 - n)]; d = np.diff(b); dt = 2.0**-n
    ito = math.sin(b[-1]) - np.sum(np.cos(b[:-1]) * d) + 0.5 * np.sum(np.sin(b[:-1])) * dt
    naive = math.sin(b[-1]) - np.sum(np.cos(b[:-1]) * d)
    print(f"n = {n:2d}: with correction {ito: .2e}   without {naive: .4f}   step^(1/2) = {2**(-n/2):.1e}")

### P4.3
**How to find it.** All three are discretisations of the same random number. The occupation estimate has a bias for large $\varepsilon$ (it averages the local times at levels in $]-\varepsilon,\varepsilon[$) and a sampling error for small $\varepsilon$; Tanaka's formula and Skorokhod's maximum agree up to the discretisation error of the sum.

In [ ]:
B = bm(1.0, 2**18)[0]; d = np.diff(B)
beta = np.concatenate([[0], np.cumsum(np.sign(B[:-1]) * d)])
print("Tanaka     :", round(abs(B[-1]) - beta[-1], 4))
print("Skorokhod  :", round(float(np.max(-beta)), 4))
for eps in (0.05, 0.02, 0.01, 0.005):
    print(f"occupation eps = {eps}:", round(float(np.sum(np.abs(B[:-1]) < eps) * 2.0**-18 / (2 * eps)), 4))
P = bm(1.0, 2**12, 4000); d = np.diff(P, axis=1)
L1 = np.abs(P[:, -1]) - np.sum(np.sign(P[:, :-1]) * d, axis=1)
A = np.abs(rng.standard_normal(4000))
for q in (0.25, 0.5, 0.75, 0.9):
    print(f"quantile {q}: L_1 {np.quantile(L1, q):.3f}   |B_1| {np.quantile(A, q):.3f}")
print("means:", round(float(L1.mean()), 3), round(float(A.mean()), 3), " sqrt(2/pi) =", round(math.sqrt(2/math.pi), 3))

### P4.4
**How to find it.** Compute $\langle M\rangle$ by a Riemann sum along each path, find the first index where it exceeds $s$, and read $M$ there. Paths on $[0,8]$ reach clock $1$ with high probability; the few that do not are discarded (a small bias).

In [ ]:
T, N, m = 8.0, 2**14, 2000
P = bm(T, N, m); dt = T / N
t = np.linspace(0, T, N + 1)
M = 0.5 * (P**2 - t)
clock = np.concatenate([np.zeros((m, 1)), np.cumsum(P[:, :-1]**2 * dt, axis=1)], axis=1)
ok = clock[:, -1] > 1
def read(s):
    idx = np.argmax(clock > s, axis=1)
    return M[np.arange(m), idx]
b_half, b_one = read(0.5)[ok], read(1.0)[ok]
print("paths kept:", ok.sum())
print("beta_1: mean", round(float(b_one.mean()), 3), " var", round(float(b_one.var()), 3), " 4th moment", round(float(np.mean(b_one**4)), 3), "(Gaussian: 0, 1, 3)")
inc = b_one - b_half
print("corr(increment, beta_1/2)  ", round(float(np.corrcoef(inc, b_half)[0, 1]), 3))
print("corr(increment, beta_1/2^2)", round(float(np.corrcoef(inc, b_half**2)[0, 1]), 3))
print("M_1 itself: skewness", round(float(np.mean((M[:, N//8] - M[:, N//8].mean())**3) / M[:, N//8].std()**3), 2))

### P4.5
**How to find it.** Simulate both martingales on a grid, take the running maximum of $|M|$ and the clock at time $1$. For $B$, $\langle B\rangle_1=1$ and $\mathbb E\sup_{s\le1}B_s^4$ is a few units; the constant $356$ is very far from sharp, as the remark after Theorem 4.18 says.

In [ ]:
P = bm(1.0, 2**12, 4000); dt = 2.0**-12
t = np.linspace(0, 1, 2**12 + 1)
supB4 = np.max(np.abs(P), axis=1)**4
print("M = B       : ratio", round(float(supB4.mean() / 1.0), 2))
Mi = 0.5 * (P**2 - t)
clock = np.sum(P[:, :-1]**2 * dt, axis=1)
print("M = int B dB: ratio", round(float(np.mean(np.max(np.abs(Mi), axis=1)**4) / np.mean(clock**2)), 2))
print("C_4 =", round(36 * (4/3)**8, 1))

### P4.6
**How to find it.** For each $n$, subsample the fine path, compute the estimator for a grid of bandwidths, and keep the bandwidth with the smallest root mean square error over the $60$ paths.

In [ ]:
nfine, m = 2**20, 60
B = bm(1.0, nfine, m)
Ltrue = np.abs(B[:, -1]) - np.sum(np.sign(B[:, :-1]) * np.diff(B, axis=1), axis=1)
for k in (8, 10, 12, 14):
    sub = B[:, ::2**(20 - k)]; dt = 2.0**-k
    errs = {}
    for h in (0.3, 0.2, 0.1, 0.05, 0.03, 0.02, 0.01, 0.005, 0.003, 0.002, 0.001):
        est = np.sum(np.abs(sub[:, :-1]) < h, axis=1) * dt / (2 * h)
        errs[h] = math.sqrt(np.mean((est - Ltrue)**2))
    h = min(errs, key=errs.get)
    print(f"n = 2^{k}: best h = {h:<6} rmse = {errs[h]:.3f}   sqrt(1/n) = {math.sqrt(dt):.4f}")

The best bandwidth shrinks with $n$, and stays within a small factor of the typical step $\sqrt{1/n}$: below that scale the samples cannot resolve the occupation, above it the estimator averages the local times of neighbouring levels. The error decreases only slowly, like a small power of $n$: local time is a very irregular functional, and it is estimated, like a density, at a rate slower than $n^{-1/2}$. Estimating occupation densities from high-frequency samples is a standard problem in the statistics of diffusions, for instance in finance (time spent near a barrier); see Jacod and Protter, *Discretization of Processes*, Springer, 2012, for the theory.